## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 02 - Paired-View YOLO ROI Adaptation |
| Model / workflow | DenseNet-121 |
| Input | 384x384, published crop 50% / YOLO ROI 50% |
| Loss | Cross-Entropy (CE) |
| Training / pipeline | Paired-view adaptation |
| Result | (filled in after the run) |

# 02 - Paired-View YOLO ROI Adaptation (DenseNet-121)

Adapts the Notebook 01 checkpoint to the input the API actually serves: expanded square YOLO ROIs.

- each training sample independently draws the published crop or the YOLO ROI
  (`ALTERNATE_VIEW_PROBABILITY`), which keeps the pre-trained signal while shifting the model
  toward the production distribution
- validation is reported on **both** domains, and the checkpoint is chosen from their mean
- the **test** split is never touched here

`BASE_CHECKPOINT` is discovered from the most recent `SELECTED_CHECKPOINT.txt` written by Notebook 01.

In [ ]:
!pip -q install "timm>=1.0" "h5py>=3.9"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

# ---- Focal CORN helpers (inlined for self-containment) -----------------
# See notebooks/_focal_corn_helpers.py for the canonical source.
NUM_CLASSES = 5
NUM_TASKS = NUM_CLASSES - 1
TASK_WEIGHTS = (1.0, 1.2, 2.0, 3.5)
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
LABEL_SMOOTHING = 0.10

def corn_loss(logits, y_train, num_classes=NUM_CLASSES, task_weights=TASK_WEIGHTS):
    loss = 0.0
    for k in range(num_classes - 1):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        targets_k = targets_k * (1 - LABEL_SMOOTHING) + (1 - targets_k) * LABEL_SMOOTHING
        w_k = task_weights[k] if k < len(task_weights) else 1.0
        loss = loss + w_k * F.binary_cross_entropy_with_logits(logits_k, targets_k)
    return loss / (num_classes - 1)

def focal_corn_loss(logits, y_train, num_classes=NUM_CLASSES, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    loss = 0.0
    for k in range(num_classes - 1):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        targets_k = targets_k * (1 - LABEL_SMOOTHING) + (1 - targets_k) * LABEL_SMOOTHING
        bce = F.binary_cross_entropy_with_logits(logits_k, targets_k, reduction="none")
        p = torch.sigmoid(logits_k)
        p_t = p * targets_k + (1 - p) * (1 - targets_k)
        focal_weight = alpha * (1 - p_t) ** gamma
        loss = loss + (focal_weight * bce).mean()
    return loss / (num_classes - 1)

def corn_probas(logits):
    cond_probas = torch.sigmoid(logits)
    batch_size = logits.size(0)
    num_classes = logits.size(1) + 1
    probas = torch.zeros(batch_size, num_classes, device=logits.device)
    cumprod = torch.cumprod(cond_probas, dim=1)
    probas[:, 0] = 1.0 - cond_probas[:, 0]
    for i in range(1, num_classes - 1):
        probas[:, i] = cumprod[:, i - 1] * (1.0 - cond_probas[:, i])
    probas[:, -1] = cumprod[:, -1]
    return probas

def corn_label_from_logits(logits):
    return torch.argmax(corn_probas(logits), dim=1)

## Configuration

`ALTERNATE_VIEW_PROBABILITY` is the one knob worth experimenting with: 0.50 is the reference value,
higher weights training toward the served distribution.

In [ ]:
# ---- reproducibility -------------------------------------------------------
SEED = 42

# ---- data ------------------------------------------------------------------
INPUT_SIZE = 384
ROTATION_DEGREES = 5
ALTERNATE_VIEW_PROBABILITY = 0.50   # P(use YOLO ROI instead of the published crop)

# ---- optimisation ----------------------------------------------------------
EPOCHS = 5
BATCH_SIZE = 48
NUM_WORKERS = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-3
PRETRAINED = False         # weights come from BASE_CHECKPOINT

# ---- paths -----------------------------------------------------------------
PUBLISHED_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "extracted/KneeXrayData/ClsKLData/kneeKL224"
)
ROI_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "derived/densenet121_yolo_square_roi_trainvaltest_v2"
)
BASE_MODEL_ROOT = Path("/content/drive/MyDrive/Models/densenet121_original_focal_corn")
MODEL_ROOT = Path("/content/drive/MyDrive/Models/densenet121_paired_roi_focal_corn")

# Read the newest pointer from Notebook 01 rather than hard-coding a timestamp.
pointers = sorted(
    BASE_MODEL_ROOT.glob("*/SELECTED_CHECKPOINT.txt"),
    key=lambda path: path.stat().st_mtime, reverse=True,
)
if not pointers:
    raise FileNotFoundError(
        f"No Notebook 01 pointer under {BASE_MODEL_ROOT}. Run 01_train_original.ipynb first."
    )
BASE_CHECKPOINT = Path(pointers[0].read_text().strip())

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_DIR = MODEL_ROOT / RUN_TIMESTAMP

for required in (PUBLISHED_ROOT, ROI_ROOT, BASE_CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(required)
RUN_DIR.mkdir(parents=True, exist_ok=False)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:   ", DEVICE)
print("Base ckpt:", BASE_CHECKPOINT)
print("Run dir:  ", RUN_DIR)

## Pair every published image with its YOLO ROI

The two folders must agree file-for-file. A missing ROI is raised immediately rather than silently
dropping a training sample.

In [ ]:
rows = []
for split in ("train", "val"):
    for grade in range(5):
        for published_path in sorted((PUBLISHED_ROOT / split / str(grade)).glob("*.png")):
            roi_path = ROI_ROOT / split / str(grade) / published_path.name
            if not roi_path.is_file():
                raise FileNotFoundError(f"Missing paired ROI: {roi_path}")
            rows.append({
                "split": split,
                "grade": grade,
                "published_path": str(published_path),
                "roi_path": str(roi_path),
            })

frame = pd.DataFrame(rows)
print(frame.groupby(["split", "grade"]).size().unstack(fill_value=0))
print("Paired records:", len(frame))

## Preprocessing, paired dataset, and model

`PairedDataset` picks the view per sample. Setting `alternate_probability` to 0.0 or 1.0 turns it
into a pure published-crop or pure ROI loader, which is how validation reports both domains.

In [ ]:
class OpenCVCLAHE:
    """LAB-space CLAHE. Identical to app/services/preprocessing_service.py."""

    def __call__(self, image_rgb):
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, a, b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a, b)), cv2.COLOR_LAB2RGB)


class SquarePad:
    """Pad to square with black borders, preserving aspect ratio."""

    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(
            image, top, side - height - top, left, side - width - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0),
        )


train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(ROTATION_DEGREES),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class PairedDataset(Dataset):
    def __init__(self, data, transform, alternate_probability):
        self.data = data.reset_index(drop=True)
        self.transform = transform
        self.alternate_probability = alternate_probability

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        use_roi = self.alternate_probability > 0 and random.random() < self.alternate_probability
        image = cv2.imread(row.roi_path if use_roi else row.published_path, cv2.IMREAD_COLOR)
        if image is None:
            raise IOError(f"Cannot read paired image at index {index}")
        return self.transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)), int(row.grade)



class DenseNet121Model(nn.Module):
    """Focal CORN variant. num_classes=4 -> 4 ordinal thresholds for 5 KL grades."""
    ARCHITECTURE = "timm_densenet121_linear_gradcam_ordinal"
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "densenet121", pretrained=PRETRAINED, num_classes=NUM_CLASSES - 1, drop_rate=0.20
        )
    @property
    def gradcam_target_layer(self):
        return self.backbone.features.norm5
    def forward(self, images):
        return self.backbone(images)
build_model = DenseNet121Model


## Fine-tune and select on both domains

`robust_selection` is the mean of the published and ROI selection scores. The per-epoch JSON is
printed so the training curve is visible in the saved notebook.

In [ ]:
def selection_score(qwk, macro_f1, macro_ap):
    """Single scalar used to pick a checkpoint. Same weighting as the reference runs."""
    return 0.55 * qwk + 0.30 * macro_f1 + 0.15 * macro_ap


def score_predictions(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1)
    _, _, macro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")
    macro_ap = average_precision_score(np.eye(5)[labels], probabilities, average="macro")
    return {
        "qwk": float(qwk),
        "macro_f1": float(macro_f1),
        "macro_ap": float(macro_ap),
        "selection": float(selection_score(qwk, macro_f1, macro_ap)),
    }


checkpoint = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)
if checkpoint.get("loss_type") not in (None, "focal_corn"):
    raise RuntimeError(f"Expected a Focal CORN checkpoint, got {checkpoint.get('loss_type')}")

model = build_model().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)

train_frame = frame[frame.split == "train"].copy()
val_frame = frame[frame.split == "val"].copy()

counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True
)
train_loader = DataLoader(
    PairedDataset(train_frame, train_transform, ALTERNATE_VIEW_PROBABILITY),
    batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")


def evaluate(data, use_roi):
    loader = DataLoader(
        PairedDataset(data, val_transform, 1.0 if use_roi else 0.0),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
    )
    model.eval()
    labels, probabilities = [], []
    with torch.inference_mode():
        for images, batch_labels in loader:
            probs = corn_probas(model(images.to(DEVICE, non_blocking=True)).float())
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs.cpu().numpy())
    return score_predictions(labels, probabilities)


best_score = -float("inf")
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, samples = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = focal_corn_loss(model(images), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * len(labels)
        samples += len(labels)
    scheduler.step()

    published = evaluate(val_frame, use_roi=False)
    roi = evaluate(val_frame, use_roi=True)
    robust = 0.5 * (published["selection"] + roi["selection"])
    row = {
        "epoch": epoch,
        "train_loss": loss_sum / samples,
        "robust_selection": robust,
        **{f"published_{k}": v for k, v in published.items()},
        **{f"roi_{k}": v for k, v in roi.items()},
    }
    history.append(row)
    print(json.dumps(row, indent=2))

    if robust > best_score:
        best_score = robust
        torch.save({
            "model_state_dict": model.state_dict(),
            "architecture": build_model.ARCHITECTURE,
            "loss_type": "focal_corn",
            "epoch": epoch,
            "input_size": INPUT_SIZE,
            "paired_view_probability": ALTERNATE_VIEW_PROBABILITY,
            "roi_expansion": 1.15,
            "robust_selection": robust,
        }, RUN_DIR / "best_model.pth")
        print(f"  => new best, robust_selection={best_score:.4f}")

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)

## Publish the checkpoint pointer

Notebook 03 reads this pointer. Copy the printed path into `.env` only after Notebook 03 confirms
the test metrics and the Grad-CAM review.

In [ ]:
checkpoint_path = RUN_DIR / "best_model.pth"
(RUN_DIR / "SELECTED_CHECKPOINT.txt").write_text(str(checkpoint_path))
(RUN_DIR / "run_config.json").write_text(json.dumps({
    "stage": "02_paired_roi",
    "base_checkpoint": str(BASE_CHECKPOINT),
    "input_size": INPUT_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "alternate_view_probability": ALTERNATE_VIEW_PROBABILITY,
    "roi_expansion": 1.15,
    "loss": "focal_corn",
    "best_robust_selection": best_score,
}, indent=2))

print("Best checkpoint:", checkpoint_path)
print("Next: run 03_evaluate_roi_test.ipynb")